# CausalTrace-X: Causal Trace-based Explainable eXfiltration Intelligence
**A causal and memory-augmented framework for detecting adaptive data exfiltration in encrypted agentic networks (no plaintext inspection).**

| Paper component | Class in this notebook | What it does |
|---|---|---|
| Encrypted Flow Feature Encoder | `FlowEncoder` | CNN + BiGRU over packet size / timing / direction sequence + session-level statistics (burst, periodicity, volume) |
| Causal Encrypted Behavior Representation (CEBR) | `CEBR` | Flow-to-agent-event cross attention with a learned temporal-causal bias (events *after* the flow cannot explain it) and a "justified volume" head |
| Hierarchical Behavior Memory Network (HBMN) | `HBMN` | Short-term (causal conv), medium-term (GRU) and long-term (slot memory with gated writes) behavioural profiles per agent |
| Counterfactual Exfiltration Analysis Module (CEAM) | `CEAM` | Compares the observed flow with a *legitimate*, an *exfiltration* and a *do(remove sensitive access)* counterfactual scenario |
| Causal Evidence Graph Transformer (CEGT) | `CEGT` | Typed-edge graph transformer over process / app / destination / session / memory / CEAM / event nodes; attention rollout gives evidence attribution |
| Adaptive Risk Calibration Mechanism (ARCM) | `ARCM` + `fit_temperature` + `adaptive_decisions` | Evidential (Dirichlet) uncertainty, temperature scaling, per-agent adaptive thresholds |

**Data note.** No public dataset contains *joint* encrypted flows and autonomous-agent event traces. This notebook therefore ships a **controlled synthetic testbed** (benign agents, burst exfiltration, low-and-slow exfiltration, behaviourally camouflaged exfiltration, hard-negative legitimate backups, held-out *unknown channels*, and traffic perturbations). Section 7 shows how to map real flow CSVs (CIC-IDS2017, ISCX-VPN, CIC-Darknet, ...) into the same tensors; the agent-event stream must come from your own agent testbed/EDR logs.

**Evaluation covered:** detection accuracy, unknown-channel generalization, robustness to traffic perturbations, latency/overhead, calibration, evidence consistency, ablations.

In [ ]:
# ===== 0. Setup =====
import os, math, time, json, random, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             average_precision_score, roc_curve, brier_score_loss)
from sklearn.ensemble import HistGradientBoostingClassifier
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# ===== Configuration =====
cfg = SimpleNamespace(
    # data
    n_train=600, n_val=100, n_test=200, n_unk=200,       # number of agents (each agent = one window of L sessions)
    L=12, P=32, E=8, S=10,                               # sessions/agent, packets kept/session, events/session, summary feats
    n_event_types=10, n_dest=64, n_proc=16, n_app=8,
    train_channels=['https', 'quic'], unk_channels=['dns', 'ws'],   # unknown channels are NEVER seen in training
    # model
    d=64, heads=4, mem_slots=8, short_k=3,
    # training
    epochs=20, bs=32, lr=2e-3, wd=1e-4, patience=6,
    lam_vol=0.5, lam_ceam=0.3, lam_rec=0.1, lam_evid=0.5,
)
EVENT_NAMES = ['file_read','file_write','net_connect','proc_spawn','api_call','db_query','archive','encrypt','clipboard','idle']
ATTACK_NAMES = {0:'benign', 1:'burst', 2:'low-and-slow', 3:'camouflaged'}

## 1. Synthetic encrypted-agentic-network testbed
Each **agent** produces `L` consecutive sessions. A session has (i) an *encrypted flow* (first `P` packets with signed size and inter-arrival time + 10 session statistics; **no payload, no channel label**) and (ii) up to `E` *agent events* (type, data sensitivity, bytes touched, time offset relative to flow start, process).

* **Benign** – API/DB work; 25 % periodic heartbeats; 10 % connections to new SaaS destinations; 20 % *hard negatives* (legitimate sensitive backups to home destinations).
* **Burst exfiltration** – sensitive reads → archive → encrypt → large upload to a new destination.
* **Low-and-slow** – small sensitive reads, periodic beaconing, volume similar to benign.
* **Camouflaged** – flow volume/timing mimics the agent's profile, but the upload is **not causally supported** by earlier data access (sensitive reads occur *after* the flow starts).
* **Unknown channels (`dns`, `ws`)** – different packet-size/timing statistics, only used at test time.

In [ ]:
# ===== Flow + session + agent generators (numpy only) =====
CH = {'https': (1200, 200, 1.0), 'quic': (1100, 250, 0.7), 'ws': (600, 150, 1.5), 'dns': (90, 120, 3.0)}  # up-pkt, down-pkt, time-scale

def gen_flow(U, channel, style, period, rng, prng=None, perturb=None):
    mu_up, mu_dn, ts = CH[channel]
    n_up = int(np.clip(U / mu_up, 2, 1500)); n_dn = int(np.clip(n_up * rng.uniform(0.4, 1.2), 2, 1500)); n = n_up + n_dn
    dirs = np.where(rng.random(n) < n_up / n, 1.0, -1.0)
    sizes = np.where(dirs > 0, rng.normal(mu_up, 0.2 * mu_up, n), rng.normal(mu_dn, 0.4 * mu_dn, n))
    sizes = np.clip(sizes, 40, 1500)
    dirs[:3] = [1, -1, -1]; sizes[:3] = [517, 1400, 200]                      # TLS-like handshake
    if style == 'burst':
        iat = rng.lognormal(np.log(0.002 * ts), 0.6, n)
    elif style == 'periodic':
        g = int(rng.integers(3, 9)); iat = rng.lognormal(np.log(0.01 * ts), 0.3, n)
        idx = (np.arange(n) % g) == 0
        iat[idx] = period * (1 + rng.normal(0, 0.03, idx.sum()))
    else:
        iat = rng.lognormal(np.log(0.05 * ts), 1.0, n)
    iat[0] = 0.0
    if perturb is not None:                                                   # adversarial / network perturbations (own RNG)
        pr = prng
        if perturb == 'padding':   sizes = np.clip(sizes + pr.uniform(0, 300, len(sizes)), 40, 1500)
        elif perturb == 'jitter':  iat = iat * pr.lognormal(0, 0.7, len(iat)) + pr.exponential(0.02, len(iat))
        elif perturb == 'delay':   iat = iat + (pr.random(len(iat)) < 0.2) * pr.uniform(0.2, 2.0, len(iat))
        elif perturb == 'drop':
            keep = pr.random(len(sizes)) > 0.1; keep[:3] = True
            sizes, dirs, iat = sizes[keep], dirs[keep], iat[keep]
        elif perturb == 'morph':   sizes = np.clip(0.5 * sizes + 0.5 * pr.uniform(60, 1400, len(sizes)), 40, 1500)
        iat[0] = 0.0; n = len(sizes)
    t = np.cumsum(iat)
    up = sizes[dirs > 0].sum(); dn = sizes[dirs < 0].sum(); dur = t[-1]
    cv = iat[1:].std() / (iat[1:].mean() + 1e-9)
    hist, _ = np.histogram(t, bins=64); fa = np.abs(np.fft.rfft(hist - hist.mean()))[1:]
    peri = fa.max() / (fa.sum() + 1e-9)
    nb = (iat > 5 * np.median(iat[1:]) + 1e-6).sum()
    summ = np.array([np.log1p(up) / 15, np.log1p(dn) / 15, np.log1p(n) / 8, np.log1p(dur) / 10, up / (up + dn + 1e-9),
                     min(cv, 10) / 5, peri, sizes.mean() / 1500, sizes.std() / 750, np.log1p(nb) / 6], dtype=np.float32)
    pkt = np.zeros((cfg.P, 3), np.float32); m = min(cfg.P, n)
    pkt[:m, 0] = sizes[:m] * dirs[:m] / 1500; pkt[:m, 1] = np.log1p(iat[:m] * 1000) / 8; pkt[:m, 2] = dirs[:m]
    return pkt, summ

def gen_session(prof, atype, rng, perturb=None, prng=None):
    E = cfg.E; ev = []          # (type, sensitivity, log-bytes, time offset [s] rel. to flow start, malicious-flag)
    btypes, bp = [0, 1, 2, 3, 4, 5, 9], [.2, .05, .1, .05, .25, .2, .15]
    def noise(k):
        for _ in range(k):
            typ = int(rng.choice(btypes, p=bp)); off = -float(rng.uniform(0, 40)) if rng.random() > 0.12 else float(rng.uniform(0, 10))
            ev.append((typ, float(rng.beta(1, 8)), float(np.clip(rng.normal(prof['read_mu'], 0.7), 3, 14)), off, 0))
    reads = lambda: sum(np.exp(e[2]) for e in ev if e[0] in (0, 5))
    base = float(np.exp(prof['mu_up'] - 0.5))
    style = 'periodic' if rng.random() < 0.25 else 'normal'
    dest = int(rng.choice(prof['home'])) if rng.random() > 0.1 else int(rng.integers(0, cfg.n_dest))
    proc = prof['proc']
    if atype == 0:
        hard = rng.random() < 0.2
        noise(int(rng.integers(2, 5 if hard else 7)))
        if hard:                                                              # legitimate sensitive backup
            for _ in range(int(rng.integers(1, 3))):
                ev.append((0, float(rng.uniform(0.4, 0.9)), float(np.clip(rng.normal(11.5, 0.5), 3, 14)), -float(rng.uniform(5, 60)), 0))
            if rng.random() < 0.5: ev.append((6, 0.5, 11.5, -float(rng.uniform(1, 5)), 0))
        U = base + reads() * float(np.exp(rng.normal(np.log(0.15), 0.4)))
    elif atype == 1:                                                          # burst exfiltration
        for _ in range(int(rng.integers(2, 5))):
            ev.append((0, float(rng.uniform(0.6, 1)), float(np.clip(rng.normal(12, 0.5), 3, 14)), -float(rng.uniform(5, 60)), 1))
        R0 = reads()
        ev.append((6, 0.9, float(np.log(R0)), -float(rng.uniform(1, 5)), 1)); ev.append((7, 0.9, float(np.log(R0)), -float(rng.uniform(0.2, 2)), 1))
        noise(int(rng.integers(1, 3)))
        U = base + reads() * float(np.exp(rng.normal(np.log(0.9), 0.15)))
        if rng.random() < 0.7: dest = int(rng.integers(40, cfg.n_dest))
        if rng.random() < 0.3: proc = int(rng.integers(8, cfg.n_proc))
        style = 'burst'
    elif atype == 2:                                                          # low-and-slow
        noise(int(rng.integers(2, 5)))
        for _ in range(int(rng.integers(1, 3))):
            ev.append((0, float(rng.uniform(0.35, 0.8)), float(np.clip(rng.normal(8, 0.5), 3, 14)), -float(rng.uniform(0, 30)), 1))
        U = base + reads() * float(np.exp(rng.normal(np.log(0.9), 0.2)))
        dest = int(rng.choice(prof['home'])) if rng.random() < 0.6 else int(rng.integers(40, cfg.n_dest))
        style = 'periodic'
    else:                                                                     # camouflaged
        noise(int(rng.integers(2, 6)))
        for _ in range(int(rng.integers(1, 3))):                              # access happens AFTER the flow starts -> not a cause
            ev.append((0, float(rng.uniform(0.3, 0.6)), float(np.clip(rng.normal(8, 0.5), 3, 14)), float(rng.uniform(1, 25)), 1))
        U = float(np.exp(rng.normal(prof['mu_up'] + 1.0, 0.2)))
        dest = int(rng.choice(prof['home'])) if rng.random() < 0.5 else int(rng.integers(0, cfg.n_dest))
    ev.sort(key=lambda e: e[3]); ev = ev[:E]
    et = np.zeros(E, np.int64); en = np.zeros((E, 3), np.float32); em = np.zeros(E, bool); ml = np.zeros(E, np.float32)
    for i, (t_, s_, b_, o_, m_) in enumerate(ev):
        et[i] = t_; en[i] = [s_, b_ / 15, np.sign(o_) * np.log1p(abs(o_)) / 5]; em[i] = True; ml[i] = m_
    pkt, summ = gen_flow(float(np.clip(U, 500, 1.2e6)), prof['channel'], style, prof['period'], rng, prng, perturb)
    return dict(pkt=pkt, summ=summ, etype=et, enum=en, emask=em, emal=ml, dest=dest, proc=proc)

def gen_agent(rng, channels, perturb=None, prng=None):
    prof = dict(home=rng.choice(40, 3, replace=False), proc=int(rng.integers(0, 8)), app=int(rng.integers(0, cfg.n_app)),
                mu_up=float(rng.normal(7.5, 0.5)), read_mu=float(rng.normal(8, 0.6)),
                period=float(rng.uniform(20, 90)), channel=str(rng.choice(channels)))
    comp = rng.random() < 0.5; atype = int(rng.integers(1, 4)) if comp else 0
    start = int(rng.integers(2, cfg.L - 3)) if comp else cfg.L
    out = []
    for t in range(cfg.L):
        a = atype if (comp and t >= start and rng.random() < 0.65) else 0
        s = gen_session(prof, a, rng, perturb, prng)
        s.update(label=int(a > 0), atype=a, app=prof['app']); out.append(s)
    return out

KEYS = ['pkt', 'summ', 'etype', 'enum', 'emask', 'emal', 'dest', 'proc', 'app', 'label', 'atype']
def build_dataset(n, seed, channels, perturb=None):
    rng = np.random.default_rng(seed); prng = np.random.default_rng(seed + 10_000)   # perturbation RNG is separate -> same underlying sessions
    out = {k: [] for k in KEYS}
    for _ in range(n):
        S = gen_agent(rng, channels, perturb, prng)
        for k in KEYS: out[k].append(np.stack([np.asarray(s[k]) for s in S]))
    return {k: np.stack(v) for k, v in out.items()}

In [ ]:
t0 = time.time()
data_train = build_dataset(cfg.n_train, 1, cfg.train_channels)
data_val   = build_dataset(cfg.n_val,   2, cfg.train_channels)
data_test  = build_dataset(cfg.n_test,  3, cfg.train_channels)
data_unk   = build_dataset(cfg.n_unk,   4, cfg.unk_channels)          # unseen channels (DNS-tunnel-like, WebSocket)
print(f"built in {time.time()-t0:.1f}s")
for nm, d in [('train', data_train), ('val', data_val), ('test', data_test), ('unknown-channel', data_unk)]:
    a = d['atype'].ravel()
    print(f"{nm:16s} sessions={a.size:6d} | exfil rate={d['label'].mean():.3f} | " +
          " ".join(f"{ATTACK_NAMES[k]}={int((a==k).sum())}" for k in range(4)))

## 2. CausalTrace-X architecture
### 2.1 Encrypted Flow Feature Encoder and agent-event encoder

In [ ]:
class FlowEncoder(nn.Module):
    """Packet-size / timing / direction sequence (CNN+BiGRU+attention pooling) fused with session-level statistics."""
    def __init__(s, d, n_summ):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(3, 32, 5, padding=2), nn.GELU(), nn.Conv1d(32, d, 3, padding=1), nn.GELU())
        s.gru = nn.GRU(d, d // 2, batch_first=True, bidirectional=True)
        s.pool = nn.Linear(d, 1)
        s.summ = nn.Sequential(nn.Linear(n_summ, d), nn.GELU(), nn.Linear(d, d))
        s.out = nn.Sequential(nn.Linear(2 * d, d), nn.LayerNorm(d))
    def forward(s, pkt, summ):
        x = s.conv(pkt.transpose(1, 2)).transpose(1, 2)
        h, _ = s.gru(x)
        w = torch.softmax(s.pool(h).squeeze(-1), -1)
        hp = (w.unsqueeze(-1) * h).sum(1)
        return s.out(torch.cat([hp, s.summ(summ)], -1)), h

class EventEncoder(nn.Module):
    """Agent action / data-access events (type, sensitivity, bytes, time offset, process) -> contextual tokens."""
    def __init__(s, cfg):
        super().__init__()
        d = cfg.d
        s.type = nn.Embedding(cfg.n_event_types, d); s.proc = nn.Embedding(cfg.n_proc, d); s.num = nn.Linear(3, d)
        layer = nn.TransformerEncoderLayer(d, cfg.heads, 2 * d, 0.1, batch_first=True, norm_first=True)
        s.enc = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
    def forward(s, et, en, em, proc):
        x = s.type(et) + s.num(en) + s.proc(proc).unsqueeze(1)
        return s.enc(x, src_key_padding_mask=~em)

### 2.2 CEBR – Causal Encrypted Behavior Representation

In [ ]:
class CEBR(nn.Module):
    """Aligns an encrypted session with the agent events that could have *caused* it.
    - cross attention (flow = query, events = keys) with a learned bias from the signed time offset
      (events that occur after the flow starts get learned penalties -> temporal causality)
    - 'justified volume' head: how many bytes the observed events can legitimately explain (trained on benign only);
      residual = observed upload - justified upload."""
    def __init__(s, d, h):
        super().__init__(); s.h = h
        s.attn = nn.MultiheadAttention(d, h, batch_first=True)
        s.lag = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, h))
        s.vol = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        s.out = nn.Sequential(nn.Linear(d + 2, d), nn.GELU(), nn.LayerNorm(d))
        s.norm = nn.LayerNorm(d)
    def forward(s, f, ev, em, toff, up_obs):
        N, E, d = ev.shape
        bias = s.lag(toff.unsqueeze(-1)).permute(0, 2, 1).masked_fill(~em.unsqueeze(1), -1e4)      # N,h,E
        o, w = s.attn(f.unsqueeze(1), ev, ev, attn_mask=bias.reshape(N * s.h, 1, E),
                      need_weights=True, average_attn_weights=False)
        o = o.squeeze(1); w = w.squeeze(2).mean(1) * em.float()                                    # N,E
        mf = em.float().unsqueeze(-1); evp = (ev * mf).sum(1) / mf.sum(1).clamp(min=1)
        vol = s.vol(evp).squeeze(-1)
        resid = (up_obs - vol).unsqueeze(-1); cos = F.cosine_similarity(f, o, dim=-1).unsqueeze(-1)
        z = s.norm(f + s.out(torch.cat([o, resid, cos], -1)))
        return z, w, torch.cat([resid, cos], -1), vol

### 2.3 HBMN – Hierarchical Behavior Memory Network

In [ ]:
class HBMN(nn.Module):
    """Per-agent behavioural memory over the session sequence.
    short  : causal conv over the last k sessions
    medium : GRU over the whole window
    long   : external slot memory, read BEFORE the current write (so it represents history), gated slot-wise writes
    A learned gate mixes the three horizons; a memory-based profile predictor yields a novelty score."""
    def __init__(s, d, slots, k=3):
        super().__init__(); s.k, s.d = k, d
        s.short = nn.Conv1d(d, d, k); s.med = nn.GRU(d, d, batch_first=True)
        s.m0 = nn.Parameter(torch.randn(slots, d) * 0.02)
        s.q, s.kk, s.v, s.g = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, 1)
        s.gate = nn.Linear(3 * d, 3); s.pred = nn.Linear(d, d)
        s.out = nn.Sequential(nn.Linear(d, d), nn.LayerNorm(d))
    def forward(s, x):                                   # x: B,L,d
        B, L, d = x.shape
        sh = s.short(F.pad(x.transpose(1, 2), (s.k - 1, 0))).transpose(1, 2)
        md, _ = s.med(x)
        M = s.m0.unsqueeze(0).expand(B, -1, -1); lg = []
        for t in range(L):
            xt = x[:, t]
            att = torch.softmax((M @ s.q(xt).unsqueeze(-1)).squeeze(-1) / math.sqrt(d), -1)
            lg.append((att.unsqueeze(-1) * M).sum(1))                                   # read (history only)
            a = torch.softmax((M @ s.kk(xt).unsqueeze(-1)).squeeze(-1) / math.sqrt(d), -1)
            w = (torch.sigmoid(s.g(xt)) * a).unsqueeze(-1)                              # write strength per slot
            M = M * (1 - w) + w * s.v(xt).unsqueeze(1)
        lg = torch.stack(lg, 1)
        gw = torch.softmax(s.gate(torch.cat([sh, md, lg], -1)), -1)
        m = s.out(gw[..., 0:1] * sh + gw[..., 1:2] * md + gw[..., 2:3] * lg)
        rec = ((s.pred(lg) - x.detach()) ** 2).mean(-1)                                 # deviation from long-term profile
        return m, torch.log1p(rec.detach()).unsqueeze(-1), gw, rec

### 2.4 CEAM – Counterfactual Exfiltration Analysis Module

In [ ]:
class CEAM(nn.Module):
    """Given event context + behavioural memory, two scenario decoders predict the flow embedding that a
    LEGITIMATE session / an EXFILTRATION session would produce. A third query is the interventional counterfactual
    do(remove sensitive-data events) under the legitimate scenario. Residuals and the log-likelihood-ratio-like
    score (e_legit - e_exfil) separate causation from mere correlation."""
    def __init__(s, d):
        super().__init__()
        mk = lambda: nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, d))
        s.leg, s.exf = mk(), mk()
        s.out = nn.Sequential(nn.Linear(3 * d + 4, d), nn.GELU(), nn.LayerNorm(d))
    def forward(s, f, ev, ev_cf, m):
        tgt = F.layer_norm(f.detach(), f.shape[-1:])
        ctx, ctx_cf = torch.cat([ev, m], -1), torch.cat([ev_cf, m], -1)
        pl, pe, pc = s.leg(ctx), s.exf(ctx), s.leg(ctx_cf)
        el = ((tgt - pl) ** 2).mean(-1, keepdim=True); ee = ((tgt - pe) ** 2).mean(-1, keepdim=True); ec = ((tgt - pc) ** 2).mean(-1, keepdim=True)
        llr = el - ee
        c = s.out(torch.cat([tgt - pl, tgt - pe, pl - pc, el, ee, ec, llr], -1))
        return c, llr, el.squeeze(-1), ee.squeeze(-1)

### 2.5 CEGT – Causal Evidence Graph Transformer

In [ ]:
class GraphAttnLayer(nn.Module):
    def __init__(s, d, h, n_edge):
        super().__init__(); s.h, s.dk = h, d // h
        s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d); s.eb = nn.Embedding(n_edge, h)
        s.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d); s.drop = nn.Dropout(0.1)
    def forward(s, x, etype, nmask):
        N, V, d = x.shape
        q, k, v = s.qkv(s.n1(x)).chunk(3, -1)
        q, k, v = [t.view(N, V, s.h, s.dk).transpose(1, 2) for t in (q, k, v)]
        att = q @ k.transpose(-1, -2) / math.sqrt(s.dk) + s.eb(etype).permute(0, 3, 1, 2)   # typed-edge bias
        att = att.masked_fill(~nmask[:, None, None, :], -1e4)
        a = s.drop(torch.softmax(att, -1))
        x = x + s.o((a @ v).transpose(1, 2).reshape(N, V, d))
        return x + s.ff(s.n2(x)), a

class CEGT(nn.Module):
    """Nodes: 0 session | 1 process | 2 application | 3 destination | 4 behavioural memory | 5 CEAM | 6.. agent events (file/db/api/...)
    Edge types: 1 event-precedes-session, 2 event-follows-session, 3 process link, 4 destination-session, 5 app-process,
                6 memory link, 7 CEAM link, 8 event-event, 9 self."""
    def __init__(s, d, h, E, layers=2):
        super().__init__()
        s.ntype = nn.Embedding(7, d); s.layers = nn.ModuleList([GraphAttnLayer(d, h, 10) for _ in range(layers)]); s.norm = nn.LayerNorm(d)
        V = 6 + E; base = torch.zeros(V, V, dtype=torch.long)
        def pair(i, j, t): base[i, j] = t; base[j, i] = t
        pair(1, 2, 5); pair(3, 0, 4); pair(1, 0, 3); pair(4, 5, 6)
        for v in (0, 1, 2, 3): pair(4, v, 6); pair(5, v, 7)
        for e in range(6, V): pair(1, e, 3); pair(0, e, 1); pair(4, e, 6); pair(5, e, 7)
        for e in range(6, V):
            for e2 in range(6, V): base[e, e2] = 8
        for i in range(V): base[i, i] = 9
        s.register_buffer('base', base)
    def forward(s, z, m, c, proc_e, app_e, dest_e, ev, em, toff):
        N, E = z.size(0), ev.size(1); dev = z.device
        x = torch.cat([torch.stack([z, proc_e, app_e, dest_e, m, c], 1), ev], 1)
        x = x + s.ntype(torch.cat([torch.arange(6, device=dev), torch.full((E,), 6, device=dev)]))
        et = s.base.unsqueeze(0).expand(N, -1, -1).clone()
        rel = 2 - (toff < 0).long(); et[:, 0, 6:] = rel; et[:, 6:, 0] = rel                # temporal-causal edge types
        nm = torch.cat([torch.ones(N, 6, dtype=torch.bool, device=dev), em], 1)
        atts = []
        for l in s.layers:
            x, a = l(x, et, nm); atts.append(a.mean(1))
        I = torch.eye(x.size(1), device=dev).unsqueeze(0); R = I
        for a in atts:                                                                      # attention rollout -> evidence attribution
            a = 0.5 * a + 0.5 * I; a = a / a.sum(-1, keepdim=True); R = a @ R
        imp = R[:, 0, 6:] * em.float(); imp = imp / imp.sum(1, keepdim=True).clamp(min=1e-9)
        return s.norm(x[:, 0]), imp

### 2.6 ARCM and the full model (+ baselines/ablations)

In [ ]:
class ARCM(nn.Module):
    """Evidential head: Dirichlet evidence -> probability + epistemic uncertainty. Calibration (temperature) and the
    per-agent adaptive threshold are applied post-hoc (fit_temperature / adaptive_decisions)."""
    def __init__(s, d, k=2):
        super().__init__()
        s.head = nn.Sequential(nn.Linear(d + 3, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, k))
    def forward(s, h, extra):
        logits = s.head(torch.cat([h, extra], -1))
        alpha = F.softplus(logits) + 1; S = alpha.sum(-1, keepdim=True)
        return logits, alpha, alpha / S, (alpha.size(-1) / S).squeeze(-1)

class CausalTraceX(nn.Module):
    def __init__(s, cfg, use_cebr=True, use_hbmn=True, use_ceam=True, use_cegt=True):
        super().__init__()
        d = cfg.d; s.cfg = cfg; s.use = dict(cebr=use_cebr, hbmn=use_hbmn, ceam=use_ceam, cegt=use_cegt)
        s.flow, s.event = FlowEncoder(d, cfg.S), EventEncoder(cfg)
        s.dest, s.proc, s.app = nn.Embedding(cfg.n_dest, d), nn.Embedding(cfg.n_proc, d), nn.Embedding(cfg.n_app, d)
        s.fuse = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.LayerNorm(d))                 # used when CEBR is off (early fusion)
        s.mix = nn.Sequential(nn.Linear(3 * d, d), nn.GELU(), nn.LayerNorm(d))                  # used when CEGT is off
        if use_cebr: s.cebr = CEBR(d, cfg.heads)
        if use_hbmn: s.hbmn = HBMN(d, cfg.mem_slots, cfg.short_k)
        if use_ceam: s.ceam = CEAM(d)
        if use_cegt: s.cegt = CEGT(d, cfg.heads, cfg.E)
        s.arcm = ARCM(d)
    def forward(s, b):
        B, L = b['label'].shape; N = B * L; d = s.cfg.d
        fl = lambda x: x.reshape(N, *x.shape[2:])
        pkt, summ, et, en, em, dest, proc, app = [fl(b[k]) for k in ('pkt', 'summ', 'etype', 'enum', 'emask', 'dest', 'proc', 'app')]
        dev = pkt.device; aux = {}
        f, _ = s.flow(pkt, summ); ev = s.event(et, en, em, proc)
        mf = em.float().unsqueeze(-1); ev_pool = (ev * mf).sum(1) / mf.sum(1).clamp(min=1)
        if s.use['cebr']:
            z, align, cex, vol = s.cebr(f, ev, em, en[..., 2], summ[:, 0]); aux['vol'] = vol
        else:
            z = s.fuse(torch.cat([f, ev_pool], -1)); align = em.float() / em.float().sum(1, keepdim=True)
            cex = torch.zeros(N, 2, device=dev)
        z = z + s.dest(dest) + s.app(app)
        if s.use['hbmn']:
            m, nov, gw, rec = s.hbmn(z.view(B, L, d)); m, nov = m.reshape(N, d), nov.reshape(N, 1); aux['rec'] = rec.reshape(N)
        else:
            m, nov = torch.zeros_like(z), torch.zeros(N, 1, device=dev)
        if s.use['ceam']:
            keep = em & ~((en[..., 0] > 0.3) | (et == 6) | (et == 7))                            # do(remove sensitive access)
            cm = keep.float().unsqueeze(-1); ev_cf = (ev * cm).sum(1) / cm.sum(1).clamp(min=1)
            c, llr, el, ee = s.ceam(f, ev_pool, ev_cf, m); aux['el'], aux['ee'] = el, ee
        else:
            c, llr = torch.zeros_like(z), torch.zeros(N, 1, device=dev)
        if s.use['cegt']:
            h, imp = s.cegt(z, m, c, s.proc(proc), s.app(app), s.dest(dest), ev, em, en[..., 2])
        else:
            h, imp = s.mix(torch.cat([z, m, c], -1)), align
        logits, alpha, prob, unc = s.arcm(h, torch.cat([nov, cex[:, :1], llr], -1))
        return dict(logits=logits, alpha=alpha, prob=prob, unc=unc, imp=imp, aux=aux)

class FlowOnly(nn.Module):
    """Baseline: encrypted-flow encoder only (no agent behaviour, no memory, no causality)."""
    def __init__(s, cfg):
        super().__init__(); s.cfg = cfg; s.flow = FlowEncoder(cfg.d, cfg.S); s.arcm = ARCM(cfg.d)
    def forward(s, b):
        B, L = b['label'].shape; N = B * L
        f, _ = s.flow(b['pkt'].reshape(N, cfg.P, 3), b['summ'].reshape(N, -1))
        logits, alpha, prob, unc = s.arcm(f, torch.zeros(N, 3, device=f.device))
        em = b['emask'].reshape(N, -1).float()
        return dict(logits=logits, alpha=alpha, prob=prob, unc=unc, imp=em / em.sum(1, keepdim=True), aux={})

m_ = CausalTraceX(cfg)
print(f"CausalTrace-X parameters: {sum(p.numel() for p in m_.parameters())/1e6:.3f} M")

## 3. Losses, training and evaluation utilities
Total loss = weighted CE + evidential (Dirichlet) loss  
&nbsp;&nbsp;+ λ<sub>vol</sub>·(CEBR justified-volume regression, benign only)  
&nbsp;&nbsp;+ λ<sub>rec</sub>·(HBMN profile prediction, benign only)  
&nbsp;&nbsp;+ λ<sub>ceam</sub>·(legit decoder on benign + exfil decoder on exfil + scenario margin).

In [ ]:
class AgentDS(Dataset):
    def __init__(s, d): s.d = d
    def __len__(s): return len(s.d['label'])
    def __getitem__(s, i): return {k: torch.as_tensor(v[i]) for k, v in s.d.items()}
def make_loader(d, bs=64, shuffle=False): return DataLoader(AgentDS(d), batch_size=bs, shuffle=shuffle)
def to_dev(b): return {k: v.to(DEVICE) for k, v in b.items()}

def kl_dirichlet(alpha):
    K = alpha.size(-1); S = alpha.sum(-1)
    return (torch.lgamma(S) - math.lgamma(K) - torch.lgamma(alpha).sum(-1)
            + ((alpha - 1) * (torch.digamma(alpha) - torch.digamma(S).unsqueeze(-1))).sum(-1))
def evidential_loss(alpha, y, ep, anneal=10):
    K = alpha.size(-1); Y = F.one_hot(y, K).float(); S = alpha.sum(-1, keepdim=True); p = alpha / S
    mse = ((Y - p) ** 2).sum(-1) + (p * (1 - p) / (S + 1)).sum(-1)
    kl = kl_dirichlet(Y + (1 - Y) * alpha)
    return (mse + min(1.0, ep / anneal) * 0.1 * kl).mean()

def compute_loss(out, b, ep):
    y = b['label'].reshape(-1); N = y.numel(); aux = out['aux']
    loss = F.cross_entropy(out['logits'], y, weight=torch.tensor([1.0, 2.0], device=y.device)) + cfg.lam_evid * evidential_loss(out['alpha'], y, ep)
    ben, atk = (y == 0).float(), (y == 1).float()
    mean_b = lambda v: (v * ben).sum() / ben.sum().clamp(min=1)
    if 'vol' in aux: loss = loss + cfg.lam_vol * mean_b((aux['vol'] - b['summ'].reshape(N, -1)[:, 0]) ** 2)
    if 'rec' in aux: loss = loss + cfg.lam_rec * mean_b(aux['rec'])
    if 'el' in aux:
        el, ee = aux['el'], aux['ee']
        l_exf = (ee * atk).sum() / atk.sum().clamp(min=1)
        l_mar = (ben * F.relu(0.2 + el - ee) + atk * F.relu(0.2 + ee - el)).mean()
        loss = loss + cfg.lam_ceam * (mean_b(el) + l_exf + l_mar)
    return loss

@torch.no_grad()
def predict(model, loader):
    model.eval(); acc = {k: [] for k in ('logits', 'prob', 'unc', 'imp')}
    for b in loader:
        o = model(to_dev(b))
        acc['logits'].append(o['logits'].cpu()); acc['prob'].append(o['prob'][:, 1].cpu()); acc['unc'].append(o['unc'].cpu()); acc['imp'].append(o['imp'].cpu())
    r = {k: torch.cat(v).numpy() for k, v in acc.items()}; ds = loader.dataset.d
    r.update(y=ds['label'].reshape(-1), atype=ds['atype'].reshape(-1), emal=ds['emal'].reshape(-1, cfg.E),
             emask=ds['emask'].reshape(-1, cfg.E), agent=np.repeat(np.arange(len(ds['label'])), cfg.L))
    return r

def train_model(model, train_d, val_d, name, epochs=None, verbose=True):
    epochs = epochs or cfg.epochs
    tl, vl = make_loader(train_d, cfg.bs, True), make_loader(val_d)
    model.to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=epochs * len(tl))
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train(); tot = 0
        for b in tl:
            b = to_dev(b); loss = compute_loss(model(b), b, ep)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); tot += loss.item()
        r = predict(model, vl); ap = average_precision_score(r['y'], r['prob']); hist.append((tot / len(tl), ap))
        if verbose and (ep % 4 == 0 or ep == epochs - 1): print(f"[{name}] ep{ep+1:02d} loss={tot/len(tl):.4f} val-AUPRC={ap:.4f}")
        if ap > best: best, bad, best_state = ap, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience: break
    model.load_state_dict(best_state); return model, hist

# ---- calibration / metrics ----
def fit_temperature(logits, y):
    lg, yy = torch.tensor(logits), torch.tensor(y); T = torch.ones(1, requires_grad=True)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(lg / T.clamp(min=0.05), yy); l.backward(); return l
    opt.step(closure); return float(T.clamp(min=0.05))
def calib_score(logits, T): return torch.softmax(torch.tensor(logits) / T, -1)[:, 1].numpy()
def best_thr(y, s):
    ts = np.linspace(0.05, 0.95, 91)
    f = [precision_recall_fscore_support(y, (s >= t).astype(int), average='binary', zero_division=0)[2] for t in ts]
    return float(ts[int(np.argmax(f))])
def det_metrics(y, s, thr):
    p = (s >= thr).astype(int); pr, rc, f1, _ = precision_recall_fscore_support(y, p, average='binary', zero_division=0)
    fpr, tpr, _ = roc_curve(y, s)
    return dict(Acc=accuracy_score(y, p), Prec=pr, Rec=rc, F1=f1, AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s), FPR_at_95TPR=float(np.interp(0.95, tpr, fpr)))
def ece(y, s, bins=15):
    conf, corr = np.maximum(s, 1 - s), ((s >= 0.5).astype(int) == y).astype(float); e = 0.0
    edges = np.linspace(0.5, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(e)
def per_attack_recall(r, s, thr): return {f"Rec[{ATTACK_NAMES[k]}]": float((s[r['atype'] == k] >= thr).mean()) for k in (1, 2, 3)}

## 4. Train CausalTrace-X, ablations and baselines

In [ ]:
VARIANTS = {
    'CausalTrace-X (full)':            lambda: CausalTraceX(cfg),
    'w/o CEBR':                        lambda: CausalTraceX(cfg, use_cebr=False),
    'w/o HBMN':                        lambda: CausalTraceX(cfg, use_hbmn=False),
    'w/o CEAM':                        lambda: CausalTraceX(cfg, use_ceam=False),
    'w/o CEGT':                        lambda: CausalTraceX(cfg, use_cegt=False),
    'Early-fusion (flow+events only)': lambda: CausalTraceX(cfg, False, False, False, False),
    'Flow-only encoder':               lambda: FlowOnly(cfg),
}
FULL = 'CausalTrace-X (full)'
R, PRED = {}, {}
def score_set(name, d, key=None):
    r = predict(R[name]['model'], make_loader(d)); r['score'] = calib_score(r['logits'], R[name]['T'])
    if key: PRED[(name, key)] = r
    return r

for name, mk in VARIANTS.items():
    set_seed(SEED); model = mk()
    t0 = time.time(); model, hist = train_model(model, data_train, data_val, name)
    rv = predict(model, make_loader(data_val)); T = fit_temperature(rv['logits'], rv['y'])
    thr = best_thr(rv['y'], calib_score(rv['logits'], T))
    R[name] = dict(model=model, T=T, thr=thr, hist=hist)
    print(f"--> {name}: trained in {time.time()-t0:.0f}s | T={T:.2f} | val-thr={thr:.2f}\n")

# classical baseline: gradient boosting on session statistics of the encrypted flow only
gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
gb.fit(data_train['summ'].reshape(-1, cfg.S), data_train['label'].ravel())
gb_thr = best_thr(data_val['label'].ravel(), gb.predict_proba(data_val['summ'].reshape(-1, cfg.S))[:, 1])
def gb_score(d): return gb.predict_proba(d['summ'].reshape(-1, cfg.S))[:, 1]

## 5. Results
### 5.1 Detection accuracy (known channels) and calibration

In [ ]:
rows = []
for name in VARIANTS:
    r = score_set(name, data_test, 'test'); s, thr = r['score'], R[name]['thr']
    row = dict(Model=name, **det_metrics(r['y'], s, thr), **per_attack_recall(r, s, thr),
               ECE=ece(r['y'], s), Brier=brier_score_loss(r['y'], s))
    rows.append(row)
yt = data_test['label'].ravel(); sg = gb_score(data_test)
rows.append(dict(Model='GradBoost (flow stats)', **det_metrics(yt, sg, gb_thr),
                 **per_attack_recall(dict(atype=data_test['atype'].ravel()), sg, gb_thr), ECE=ece(yt, sg), Brier=brier_score_loss(yt, sg)))
res_main = pd.DataFrame(rows).set_index('Model').round(4)
display(res_main)

### 5.2 Unknown-channel generalization (DNS-tunnel-like and WebSocket channels never seen in training)

In [ ]:
rows = []
for name in VARIANTS:
    r = score_set(name, data_unk, 'unk'); s, thr = r['score'], R[name]['thr']
    auc_known = res_main.loc[name, 'AUROC']; m = det_metrics(r['y'], s, thr)
    rows.append(dict(Model=name, **m, **per_attack_recall(r, s, thr), AUROC_gap=auc_known - m['AUROC']))
yu = data_unk['label'].ravel(); su = gb_score(data_unk)
rows.append(dict(Model='GradBoost (flow stats)', **det_metrics(yu, su, gb_thr), **per_attack_recall(dict(atype=data_unk['atype'].ravel()), su, gb_thr),
                 AUROC_gap=res_main.loc['GradBoost (flow stats)', 'AUROC'] - roc_auc_score(yu, su)))
res_unk = pd.DataFrame(rows).set_index('Model').round(4); display(res_unk)

### 5.3 Robustness to traffic perturbations (padding, jitter, delay injection, packet drop, size morphing)

In [ ]:
PERT = ['padding', 'jitter', 'delay', 'drop', 'morph']
pert_sets = {p: build_dataset(cfg.n_test, 3, cfg.train_channels, perturb=p) for p in PERT}   # same sessions as clean test, perturbed flows
rob_f1, rob_auc = {}, {}
for name in VARIANTS:
    rob_f1[name], rob_auc[name] = {'clean': res_main.loc[name, 'F1']}, {'clean': res_main.loc[name, 'AUROC']}
    for p in PERT:
        r = score_set(name, pert_sets[p], ('pert', p)); m = det_metrics(r['y'], r['score'], R[name]['thr'])
        rob_f1[name][p], rob_auc[name][p] = m['F1'], m['AUROC']
rob_f1['GradBoost (flow stats)'], rob_auc['GradBoost (flow stats)'] = {'clean': res_main.loc['GradBoost (flow stats)', 'F1']}, {'clean': res_main.loc['GradBoost (flow stats)', 'AUROC']}
for p in PERT:
    m = det_metrics(pert_sets[p]['label'].ravel(), gb_score(pert_sets[p]), gb_thr)
    rob_f1['GradBoost (flow stats)'][p], rob_auc['GradBoost (flow stats)'][p] = m['F1'], m['AUROC']
rob_f1, rob_auc = pd.DataFrame(rob_f1).T.round(4), pd.DataFrame(rob_auc).T.round(4)
print("F1 under perturbation"); display(rob_f1); print("AUROC under perturbation"); display(rob_auc)

fig, ax = plt.subplots(figsize=(8, 3.6)); im = ax.imshow(rob_f1.values, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(rob_f1.shape[1])); ax.set_xticklabels(rob_f1.columns); ax.set_yticks(range(rob_f1.shape[0])); ax.set_yticklabels(rob_f1.index, fontsize=8)
for i in range(rob_f1.shape[0]):
    for j in range(rob_f1.shape[1]): ax.text(j, i, f"{rob_f1.values[i,j]:.2f}", ha='center', va='center', color='w', fontsize=8)
ax.set_title("F1 under traffic perturbations"); plt.colorbar(im); plt.tight_layout(); plt.show()

### 5.4 Detection latency and computational overhead

In [ ]:
def efficiency(model, d, n_rep=5):
    model.eval(); b = to_dev(next(iter(make_loader(d, 64)))); n_sess = b['label'].numel()
    with torch.no_grad():
        for _ in range(2): model(b)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_rep): model(b)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) / n_rep
    flops = None
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(display=False) as fc, torch.no_grad(): model(b)
        flops = fc.get_total_flops() / n_sess / 1e6
    except Exception: pass
    return dict(Params_M=sum(p.numel() for p in model.parameters()) / 1e6, Latency_ms_per_session=1000 * dt / n_sess, MFLOPs_per_session=flops)
eff = pd.DataFrame({n: efficiency(R[n]['model'], data_test) for n in VARIANTS}).T.round(4); display(eff)

### 5.5 Calibration (ARCM): reliability diagram before/after temperature scaling and uncertainty quality

In [ ]:
r = PRED[(FULL, 'test')]; y = r['y']; raw = torch.softmax(torch.tensor(r['logits']), -1)[:, 1].numpy(); cal = r['score']
def reliability(y, s, bins=10):
    e = np.linspace(0, 1, bins + 1); xs, ys = [], []
    for lo, hi in zip(e[:-1], e[1:]):
        m = (s >= lo) & (s < hi + (1e-9 if hi == 1 else 0))
        if m.sum() >= 5: xs.append(s[m].mean()); ys.append(y[m].mean())
    return xs, ys
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot([0, 1], [0, 1], 'k--', lw=1)
for lab, s_ in [('raw softmax', raw), ('temperature-scaled', cal), ('evidential', r['prob'])]:
    xs, ys = reliability(y, s_); ax[0].plot(xs, ys, 'o-', label=f"{lab} (ECE={ece(y, s_):.3f})")
ax[0].set_xlabel('predicted P(exfil)'); ax[0].set_ylabel('empirical frequency'); ax[0].legend(fontsize=8); ax[0].set_title('Reliability')
ax[1].hist(r['unc'][(y == r['y']) & ((cal >= R[FULL]['thr']) == y)], bins=30, alpha=.6, label='correct'); ax[1].hist(r['unc'][(cal >= R[FULL]['thr']) != y], bins=30, alpha=.6, label='wrong')
ax[1].set_title('Evidential uncertainty'); ax[1].legend(); ax[1].set_xlabel('u = K / S')
order = np.argsort(r['unc']); fr = np.linspace(0.2, 1, 9); accs = []
pred_lab = (cal >= R[FULL]['thr']).astype(int)
for f_ in fr: idx = order[:int(f_ * len(order))]; accs.append((pred_lab[idx] == y[idx]).mean())
ax[2].plot(fr, accs, 'o-'); ax[2].set_xlabel('fraction kept (lowest uncertainty first)'); ax[2].set_ylabel('accuracy'); ax[2].set_title('Confidence-aware abstention')
plt.tight_layout(); plt.show()
print(f"temperature T={R[FULL]['T']:.3f}; ECE raw={ece(y, raw):.4f} -> calibrated={ece(y, cal):.4f}")

# ---- per-agent adaptive thresholds (ARCM) ----
def adaptive_decisions(score, thr0, g_benign, gamma=0.5, beta=0.8):
    S = score.reshape(-1, cfg.L); pred = np.zeros_like(S, dtype=int)
    for a in range(S.shape[0]):
        ema = g_benign
        for t in range(cfg.L):
            thr = float(np.clip(thr0 + gamma * (ema - g_benign), 0.1, 0.95)); pred[a, t] = int(S[a, t] >= thr)
            if pred[a, t] == 0: ema = beta * ema + (1 - beta) * S[a, t]       # agent baseline learned from accepted-benign sessions
    return pred.reshape(-1)
rv = predict(R[FULL]['model'], make_loader(data_val)); sv = calib_score(rv['logits'], R[FULL]['T']); g_b = float(sv[rv['y'] == 0].mean())
fixed = (cal >= R[FULL]['thr']).astype(int); adapt = adaptive_decisions(cal, R[FULL]['thr'], g_b)
def prf(y, p): pr, rc, f1, _ = precision_recall_fscore_support(y, p, average='binary', zero_division=0); return dict(Prec=pr, Rec=rc, F1=f1, FPR=float(((p == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)))
display(pd.DataFrame({'fixed threshold': prf(y, fixed), 'adaptive per-agent threshold': prf(y, adapt)}).T.round(4))

### 5.6 Evidence attribution: localization, consistency and stability

In [ ]:
def evidence_metrics(r_clean, r_pert=None, k=3, imp=None):
    imp = r_clean['imp'] if imp is None else imp; em, emal = r_clean['emask'], r_clean['emal']; sel = (r_clean['y'] == 1)
    top = np.argsort(-np.where(em, imp, -1), axis=1)[:, :k]; hit = np.take_along_axis(emal, top, 1)
    nm = emal.sum(1); p_at = hit.sum(1) / np.minimum(k, np.maximum(nm, 1))
    out = {'Hit@3 (any malicious event in top-3)': float((hit.sum(1) > 0)[sel].mean()), 'Prec@3 (normalised)': float(p_at[sel].mean())}
    if r_pert is not None:
        pi = r_pert['imp'] if imp is r_clean['imp'] else None
        if pi is not None:
            topp = np.argsort(-np.where(em, pi, -1), axis=1)[:, :k]
            jac = np.array([len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(top, topp)])
            out['Stability (Jaccard@3, clean vs padded)'] = float(jac[sel].mean())
    return out
ev_rows = {}
for name in [FULL, 'w/o CEGT', 'w/o CEBR', 'Early-fusion (flow+events only)']:
    ev_rows[name] = evidence_metrics(PRED[(name, 'test')], PRED[(name, ('pert', 'padding'))])
rng_ = np.random.default_rng(0); rnd = dict(PRED[(FULL, 'test')]); rnd_imp = rng_.random(rnd['imp'].shape)
ev_rows['Random attribution'] = evidence_metrics(rnd, None, imp=rnd_imp)
res_ev = pd.DataFrame(ev_rows).T.round(4); display(res_ev)

# per-attack-type localization for the full model
r = PRED[(FULL, 'test')]
for k in (1, 2, 3):
    m = r['atype'] == k; sub = {kk: (vv[m] if hasattr(vv, '__len__') and len(vv) == len(m) else vv) for kk, vv in r.items()}
    print(f"{ATTACK_NAMES[k]:13s}", {a: round(b, 3) for a, b in evidence_metrics(dict(sub, y=np.ones(m.sum(), int))).items()})

### 5.7 Ablation study

In [ ]:
abl = res_main.loc[[FULL, 'w/o CEBR', 'w/o HBMN', 'w/o CEAM', 'w/o CEGT', 'Early-fusion (flow+events only)', 'Flow-only encoder'],
                   ['F1', 'AUROC', 'AUPRC', 'FPR_at_95TPR', 'Rec[burst]', 'Rec[low-and-slow]', 'Rec[camouflaged]', 'ECE']].copy()
abl['AUROC_unknown_ch'] = res_unk.loc[abl.index, 'AUROC']; abl['F1_padding'] = rob_f1.loc[[i for i in abl.index if i in rob_f1.index], 'padding'].reindex(abl.index)
display(abl.round(4))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6)); names = list(abl.index); xs = np.arange(len(names))
ax[0].bar(xs, abl['F1']); ax[0].set_title('F1 (known channels)')
w = 0.27
for i, c in enumerate(['Rec[burst]', 'Rec[low-and-slow]', 'Rec[camouflaged]']): ax[1].bar(xs + (i - 1) * w, abl[c], w, label=c)
ax[1].set_title('Recall by attack type'); ax[1].legend(fontsize=7)
ax[2].bar(xs - 0.2, abl['AUROC'], 0.4, label='known'); ax[2].bar(xs + 0.2, abl['AUROC_unknown_ch'], 0.4, label='unknown channels'); ax[2].set_title('AUROC'); ax[2].legend(fontsize=8)
for a in ax: a.set_xticks(xs); a.set_xticklabels([n.replace(' (', '\n(') for n in names], rotation=60, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

### 5.8 ROC curves and an explainable case study

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, style in [(FULL, '-'), ('Early-fusion (flow+events only)', '--'), ('Flow-only encoder', ':')]:
    r = PRED[(name, 'test')]; fpr, tpr, _ = roc_curve(r['y'], r['score']); ax[0].plot(fpr, tpr, style, label=f"{name} (AUC={roc_auc_score(r['y'], r['score']):.3f})")
fpr, tpr, _ = roc_curve(yt, sg); ax[0].plot(fpr, tpr, '-.', label=f"GradBoost (AUC={roc_auc_score(yt, sg):.3f})")
ax[0].plot([0, 1], [0, 1], 'k--', lw=.5); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(fontsize=8); ax[0].set_title('ROC (known channels)')
for name, style in [(FULL, '-'), ('Flow-only encoder', ':')]:
    r = PRED[(name, 'unk')]; fpr, tpr, _ = roc_curve(r['y'], r['score']); ax[1].plot(fpr, tpr, style, label=f"{name} (AUC={roc_auc_score(r['y'], r['score']):.3f})")
fpr, tpr, _ = roc_curve(yu, su); ax[1].plot(fpr, tpr, '-.', label=f"GradBoost (AUC={roc_auc_score(yu, su):.3f})")
ax[1].plot([0, 1], [0, 1], 'k--', lw=.5); ax[1].set_title('ROC (unknown channels)'); ax[1].legend(fontsize=8); ax[1].set_xlabel('FPR')
plt.tight_layout(); plt.show()

def explain_session(idx, key='test', data=None):
    """Human-readable evidence report for one session of the full model."""
    data = data or data_test; r = PRED[(FULL, key)]; E = cfg.E
    et, en, em = data['etype'].reshape(-1, E)[idx], data['enum'].reshape(-1, E, 3)[idx], data['emask'].reshape(-1, E)[idx]
    mal = data['emal'].reshape(-1, E)[idx]; imp = r['imp'][idx]
    print(f"session {idx} | true={ATTACK_NAMES[int(r['atype'][idx])]} | P(exfil)={r['score'][idx]:.3f} | evidential uncertainty={r['unc'][idx]:.3f} | decision={'ALERT' if r['score'][idx] >= R[FULL]['thr'] else 'allow'}")
    print(f"observed upload feature={data['summ'].reshape(-1, cfg.S)[idx][0]:.3f}")
    names, vals, cols = [], [], []
    for i in range(E):
        if not em[i]: continue
        toff = np.sign(en[i][2]) * np.expm1(abs(en[i][2]) * 5)
        names.append(f"{EVENT_NAMES[et[i]]}\nsens={en[i][0]:.2f} t={toff:+.0f}s"); vals.append(imp[i]); cols.append('crimson' if mal[i] else 'steelblue')
    fig, ax = plt.subplots(figsize=(9, 3)); ax.bar(range(len(vals)), vals, color=cols); ax.set_xticks(range(len(vals))); ax.set_xticklabels(names, fontsize=7)
    ax.set_title('Evidence attribution (CEGT rollout). Red = ground-truth malicious event'); plt.tight_layout(); plt.show()

for k in (2, 3):
    cand = np.where((PRED[(FULL, 'test')]['atype'] == k) & (PRED[(FULL, 'test')]['score'] >= R[FULL]['thr']))[0]
    if len(cand): explain_session(int(cand[0]))

## 6. Save results and model

In [ ]:
os.makedirs('causaltrace_outputs', exist_ok=True)
res_main.to_csv('causaltrace_outputs/detection_accuracy.csv'); res_unk.to_csv('causaltrace_outputs/unknown_channel.csv')
rob_f1.to_csv('causaltrace_outputs/robustness_f1.csv'); rob_auc.to_csv('causaltrace_outputs/robustness_auroc.csv')
eff.to_csv('causaltrace_outputs/efficiency.csv'); res_ev.to_csv('causaltrace_outputs/evidence_consistency.csv'); abl.to_csv('causaltrace_outputs/ablation.csv')
torch.save({'state_dict': R[FULL]['model'].state_dict(), 'T': R[FULL]['T'], 'thr': R[FULL]['thr'], 'cfg': vars(cfg)}, 'causaltrace_outputs/causaltrace_x.pt')
print(os.listdir('causaltrace_outputs'))

## 7. Plugging in real encrypted-traffic datasets
`FlowEncoder` expects, per session, (a) the first `P=32` packets as `[signed_size/1500, log1p(iat_ms)/8, direction]` and (b) 10 session statistics. From **pcaps** you can compute both exactly (use `scapy`/`pyshark`/`nfstream`). From **CICFlowMeter CSVs** only the statistics are available, so the helper below fills `summ` and leaves the packet sequence for you to supply.

The agent-event stream (`etype`, `enum`, `emask`, `proc`, `app`) must be collected from the agent runtime (tool-call logs, EDR/eBPF file-access events, process trees) and time-aligned with each flow; public encrypted-traffic datasets do not contain it. Suggested labels: replay public exfiltration/C2 flows (CIC-IDS2017/2018, CTU-13, CIC-Darknet, ISCX-VPN) into your agent testbed while the agent performs scripted benign/malicious tasks.

In [ ]:
def flows_csv_to_summary(df):
    """CICFlowMeter-style columns -> the 10 summary features used by FlowEncoder (periodicity cannot be recovered from aggregates -> 0)."""
    df = df.copy(); df.columns = [c.strip() for c in df.columns]
    g = lambda c: df[c].astype(float).values if c in df.columns else np.zeros(len(df))
    up, dn = g('Total Length of Fwd Packets'), g('Total Length of Bwd Packets')
    n = g('Total Fwd Packets') + g('Total Backward Packets'); dur = g('Flow Duration') / 1e6
    cv = np.clip(g('Flow IAT Std') / (g('Flow IAT Mean') + 1e-9), 0, 10)
    summ = np.stack([np.log1p(up) / 15, np.log1p(dn) / 15, np.log1p(n) / 8, np.log1p(dur) / 10, up / (up + dn + 1e-9), cv / 5,
                     np.zeros(len(df)), g('Packet Length Mean') / 1500, g('Packet Length Std') / 750, np.zeros(len(df))], 1).astype(np.float32)
    return summ
# Example:  summ = flows_csv_to_summary(pd.read_csv('path/to/flows.csv'));  then group sessions per agent into windows of cfg.L
#           and fill the dict keys in KEYS (pkt, summ, etype, enum, emask, emal, dest, proc, app, label, atype) -> train_model(...)

## 8. Notes for the paper
* All numbers above come from the synthetic testbed; report mean ± std over several seeds (wrap Section 4 in a loop over `SEED`) and, when possible, over real agent-instrumented traces.
* Evidence-consistency metrics use *ground-truth malicious event flags*, available only in the testbed; on real data use analyst-labelled events or replay-based interventions.
* `CEAM` counterfactuals are **model-based** (learned scenario decoders), not identified causal effects; state this assumption explicitly.
* Ethical use: this is a defensive detector; it contains no exfiltration tooling, only a statistical traffic generator.